# Stage 13: Classification Analysis
**Project**: Cyber Crime Analytics for National Security  
**Syllabus Alignment**: Unit 5 — Classification (Classification vs. Prediction, Decision Trees, Bayesian Classification: Naive Bayes, Support Vector Machines: Linear & RBF Kernels, Ensemble Methods: Random Forest, Classification Metrics: Accuracy, Precision, Recall, Specificity, F1-Score, ROC-AUC, Confusion Matrix, Leakage Prevention)  
**Data Source**: Longitudinal State/UT Historical Panel (2018–2022 Rajya Sabha Cybercrime Series)  
**Sample Size**: $N = 106$ Observations ($N_{\text{train}} = 70$ for target years 2020 & 2021; $N_{\text{test}} = 36$ for held-out target year 2022)  

---

## 1. Objective & Analytical Question

### Analytical Question:
> **"Can historical State/UT cybercrime volume patterns classify whether the following year's aggregate cybercrime volume belongs to a high-volume regime?"**

### Theoretical & Methodological Context:
- **Classification vs. Regression (Stage 7 vs. Stage 13)**: Whereas Stage 7 estimated the continuous future volume ($\hat{Y}_{t} \in \mathbb{R}^+$), Stage 13 addresses categorical regime assignment ($Y_t \in \{0, 1\}$, Low vs. High volume regime).
- **Training-Derived Threshold Exclusivity**: To prevent test leakage and threshold-tuning bias, the binary target boundary is computed strictly from the **training partition median** ($\text{Threshold} = 367.0$ cases). A state in year $t$ is labeled $\text{HIGH\_NEXT\_YEAR} = 1$ if $y_t \ge 367.0$, else $0$.
- **Strict Chronological Horizon**: Predictors are constructed strictly from years $t-1$ and $t-2$. Zero contemporaneous target-year features or 2023 detailed category features are permitted.
- **Pipelines & Leakage Prevention**: All feature normalization (`StandardScaler`) is fitted strictly on training data within scikit-learn Pipelines.
- **Academic Non-Causal Framing**: Features capture historical volume scale and momentum. High classification accuracy reflects jurisdictional temporal persistence rather than causal determinants.


In [1]:
# Setup environment, imports, and display configurations
import os
import sys
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

from src.classification import (
    load_and_construct_classification_panel,
    run_leakage_audit,
    get_class_distributions,
    build_classification_models,
    evaluate_models,
    get_feature_importances,
    plot_classification_figures,
    save_stage13_tables,
    FEATURE_COLS
)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Working Directory: {project_root}")
print(f"Predictor Feature Vector: {FEATURE_COLS}")


Python Version: 3.13.7
Working Directory: C:\Users\kaval vyas\OneDrive\Desktop\Projects\cybercrime-analytics
Predictor Feature Vector: ['lag_1', 'lag_2', 'lag_diff', 'lag_growth_rate', 'log_lag_1', 'log_lag_2']


## 2. Longitudinal Panel Construction & Target Definition

The longitudinal dataset constructs records of $(State/UT, \text{target\_year})$ with lagged historical indicators:
- **Target Year 2020** (Train): Predictors from 2019 ($t-1$) and 2018 ($t-2$). $N = 35$ (Ladakh missing in 2018/19).
- **Target Year 2021** (Train): Predictors from 2020 ($t-1$) and 2019 ($t-2$). $N = 35$.
- **Target Year 2022** (Held-Out Test): Predictors from 2021 ($t-1$) and 2020 ($t-2$). $N = 36$ (Ladakh present in 2020/21).

### Target Threshold Derivation:
$$\text{Threshold} = \text{Median}(Y_{\text{train}}) = 367.0 \text{ cases}$$
$$\text{HIGH\_NEXT\_YEAR} = \begin{cases} 1 & \text{if } y_t \ge 367.0 \\ 0 & \text{if } y_t < 367.0 \end{cases}$$


In [2]:
# Construct panel dataset and derive training threshold
panel_df, threshold = load_and_construct_classification_panel('data/processed/trend_2018_2022.csv')

print(f"Total Longitudinal Panel Size: {len(panel_df)} observations")
print(f"Training Partition (2020 & 2021): N = {(panel_df['split'] == 'train').sum()}")
print(f"Held-Out Test Partition (2022): N = {(panel_df['split'] == 'test').sum()}")
print(f"Training Median Threshold: {threshold:.1f} cases\n")

display(panel_df.head(10))


Total Longitudinal Panel Size: 106 observations
Training Partition (2020 & 2021): N = 70
Held-Out Test Partition (2022): N = 36
Training Median Threshold: 367.0 cases



,state_name,target_year,split,lag_1,lag_2,lag_diff,lag_growth_rate,log_lag_1,log_lag_2,target_actual,HIGH_NEXT_YEAR
0,Andhra Pradesh,2020,train,1886.0,1207.0,679.0,0.562086,7.542744,7.096721,1899.0,1
1,Andhra Pradesh,2021,train,1899.0,1886.0,13.0,0.006889,7.549609,7.542744,1875.0,1
2,Andhra Pradesh,2022,test,1875.0,1899.0,-24.0,-0.012632,7.536897,7.549609,2341.0,1
3,Arunachal Pradesh,2020,train,8.0,7.0,1.0,0.125000,2.197225,2.079442,30.0,0
4,Arunachal Pradesh,2021,train,30.0,8.0,22.0,2.444444,3.433987,2.197225,47.0,0
5,Arunachal Pradesh,2022,test,47.0,30.0,17.0,0.548387,3.871201,3.433987,14.0,0
6,Assam,2020,train,2231.0,2022.0,209.0,0.103312,7.710653,7.612337,3530.0,1
7,Assam,2021,train,3530.0,2231.0,1299.0,0.581989,8.169336,7.710653,4846.0,1
8,Assam,2022,test,4846.0,3530.0,1316.0,0.372699,8.486115,8.169336,1733.0,1
9,Bihar,2020,train,1050.0,374.0,676.0,1.802667,6.957497,5.926926,1512.0,1


## 3. Class Balance & Partition Distributions

To prevent majority-class bias and ensure transparent evaluation, we inspect class distributions across training and test splits.


In [3]:
# Compute class distributions
class_dist_df = get_class_distributions(panel_df, threshold)
display(class_dist_df)


,Partition,Total_Obs,Class_0_Low,Class_1_High,Pct_Class_0,Pct_Class_1,Threshold_Applied
0,Full Panel,106,51,55,48.11,51.89,367.0
1,Training (2020-2021),70,35,35,50.00,50.00,367.0
2,Held-Out Test (2022),36,16,20,44.44,55.56,367.0


## 4. Formal Leakage and Validity Audit

We execute a formal 7-point leakage audit to guarantee mathematical and methodological integrity.


In [4]:
# Run formal 7-point leakage audit
audit_df = run_leakage_audit(panel_df, threshold)
display(audit_df)


,Check,Status,Detail
0,Target Leakage Prevention,PASSED,Neither target_actual nor HIGH_NEXT_YEAR is in the feature predictor list.
1,Temporal Horizon Independence,PASSED,All features are lagged (t-1 and t-2 relative to target_year). Zero contemporaneous features.
2,2023 Feature Exclusion,PASSED,Zero 2023 sectional/motive attributes present in the historical panel dataset.
3,Threshold Training Exclusivity,PASSED,High-volume threshold (367.0 cases) derived solely from 70 training observations.
4,Chronological Split Integrity,PASSED,"Training target years: [2020, 2021] (N=70), Held-out test target year: [2022] (N=36)."
5,Observation Tuple Uniqueness,PASSED,"All 106 (State/UT, target_year) tuples are strictly unique."
6,Longitudinal Panel Structure,PASSED,Jurisdictions repeat across target years in a longitudinal panel; temporal order is strictly maintained without cross-period shuffling.


## 5. Model Training & Evaluation Suite

We train and compare 6 classification architectures spanning Unit 5:
1. **Baseline Classifier**: Majority Class (`DummyClassifier(strategy='most_frequent')`).
2. **Decision Tree Classifier**: Interpretable tree with `max_depth=3` and Gini impurity.
3. **Gaussian Naive Bayes**: Probabilistic classifier assuming conditional feature independence given regime class.
4. **Linear Support Vector Machine**: Linear hyperplane classifier in standardized feature space (`StandardScaler -> SVC(kernel='linear')`).
5. **RBF Support Vector Machine**: Nonlinear kernel boundary classifier (`StandardScaler -> SVC(kernel='rbf')`).
6. **Random Forest Classifier**: Constrained bootstrap ensemble of 100 decision trees.


In [5]:
# Build, train, and evaluate classification models
models = build_classification_models()
metrics_df, cm_df, prob_dict = evaluate_models(models, panel_df)

print("=== CLASSIFICATION PERFORMANCE METRICS (HELD-OUT 2022 TEST HORIZON) ===")
display(metrics_df)

print("\n=== CONFUSION MATRICES SUMMARY (N_test = 36) ===")
display(cm_df)


=== CLASSIFICATION PERFORMANCE METRICS (HELD-OUT 2022 TEST HORIZON) ===

=== CONFUSION MATRICES SUMMARY (N_test = 36) ===


,Model,Accuracy,Balanced_Accuracy,Precision,Recall,Specificity,F1_Score,ROC_AUC
0,Baseline (Most Frequent),0.4444,0.500,0.0,0.00,1.0,0.0000,0.500
1,Decision Tree (depth=3),0.9722,0.975,1.0,0.95,1.0,0.9744,0.975
2,Gaussian Naive Bayes,1.0000,1.000,1.0,1.00,1.0,1.0000,1.000
3,Linear SVM,1.0000,1.000,1.0,1.00,1.0,1.0000,1.000
4,RBF SVM,1.0000,1.000,1.0,1.00,1.0,1.0000,1.000
5,Random Forest,1.0000,1.000,1.0,1.00,1.0,1.0000,1.000


,Model,True_Negative_TN,False_Positive_FP,False_Negative_FN,True_Positive_TP,Total_Test_Obs
0,Baseline (Most Frequent),16,0,20,0,36
1,Decision Tree (depth=3),16,0,1,19,36
2,Gaussian Naive Bayes,16,0,0,20,36
3,Linear SVM,16,0,0,20,36
4,RBF SVM,16,0,0,20,36
5,Random Forest,16,0,0,20,36


## 6. Decision Tree Hyperparameter Sensitivity

We evaluate Decision Tree performance across depth constraints $\text{max\_depth} \in \{2, 3, 4\}$ on the training partition.


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

train_mask = panel_df['split'] == 'train'
test_mask = panel_df['split'] == 'test'
X_tr = panel_df.loc[train_mask, FEATURE_COLS]
y_tr = panel_df.loc[train_mask, 'HIGH_NEXT_YEAR']
X_te = panel_df.loc[test_mask, FEATURE_COLS]
y_te = panel_df.loc[test_mask, 'HIGH_NEXT_YEAR']

tree_sens = []
for d in [2, 3, 4]:
    dt_cand = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_cand.fit(X_tr, y_tr)
    p_te = dt_cand.predict(X_te)
    tree_sens.append({
        'Max_Depth': d,
        'Leaves': dt_cand.get_n_leaves(),
        'Test_Accuracy': round(accuracy_score(y_te, p_te), 4),
        'Test_F1': round(f1_score(y_te, p_te), 4),
        'Top_Split_Feature': FEATURE_COLS[np.argmax(dt_cand.feature_importances_)]
    })

display(pd.DataFrame(tree_sens))


,Max_Depth,Leaves,Test_Accuracy,Test_F1,Top_Split_Feature
0,2,3,0.9722,0.9744,log_lag_1
1,3,4,0.9722,0.9744,log_lag_1
2,4,4,0.9722,0.9744,log_lag_1


## 7. Feature Importance & Temporal Persistence

We extract Gini feature importances from the Tree-based architectures and analyze the role of historical scale vs momentum.


In [7]:
# Extract and display feature importances
feature_imp_df = get_feature_importances(models, panel_df)
display(feature_imp_df)


,Feature,Decision_Tree_Importance,Random_Forest_Importance
0,log_lag_1,0.9444,0.2714
1,log_lag_2,0.0556,0.2542
2,lag_2,0.0000,0.2372
3,lag_1,0.0000,0.2126
4,lag_diff,0.0000,0.0233
5,lag_growth_rate,0.0000,0.0013


## 8. Visualizations & Analytical Figures

We generate and render the Stage 13 visualization suite (Figures 40 to 44).


In [8]:
# Generate and save figures and tables
plot_classification_figures(
    panel_df, class_dist_df, metrics_df, cm_df, prob_dict, models, feature_imp_df
)
save_stage13_tables(panel_df, class_dist_df, metrics_df, cm_df, feature_imp_df, audit_df)
print("Visualizations (Figures 40-44) and Tables successfully generated.")


Visualizations (Figures 40-44) and Tables successfully generated.


## 9. Comparison: Stage 7 Regression vs. Stage 13 Classification

| Dimension | Stage 7 — Regression | Stage 13 — Classification |
| :--- | :--- | :--- |
| **Analytical Question** | "How many cybercrime cases will occur in State $s$ in year $t$?" | "Will State $s$ in year $t$ belong to a high-volume regime?" |
| **Target Variable ($Y$)** | Continuous volume $\hat{Y}_t \in [0, \infty)$ | Binary category $Y_t \in \{0, 1\}$ ($\ge 367$ cases) |
| **Evaluation Metrics** | $R^2$, MAE, RMSE, MedAE | Accuracy, Precision, Recall, Specificity, F1, ROC-AUC |
| **Best Model** | Log-Linear OLS ($R^2 = 0.8878$, $\text{MAE} = 459.7$) | Linear SVM / Random Forest / Naive Bayes ($\text{Acc} = 1.0, F_1 = 1.0$) |
| **Analytical Utility** | Precise volumetric forecasting for budget allocation | Regime threshold triaging for macro resource tiering |

---

## 10. Limitations & Academic Conclusions

1. **Small-$N$ Constraint**: The test horizon contains exactly $N = 36$ jurisdictions. While accuracy is high, small sample sizes preclude claims of universal generalizability.
2. **Temporal Volume Persistence**: Near-perfect regime separation arises because Indian state cybercrime volume is heavily concentrated in large populated states (Maharashtra, Telangana, Karnataka, UP) and extremely low in small UTs/northeastern states. The model detects historical volume inertia rather than criminological causation.
3. **Threshold Sensitivity**: The threshold is objectively derived as the training median ($367.0$ cases). Alternative quantile thresholds would shift class boundaries.
4. **Syllabus Educational Purpose**: This stage successfully demonstrates Unit 5 machine learning classification algorithms under strict leakage prevention, pipeline normalization, and chronological splitting.
